# AgentCore Online Evaluation

Researcher Agent invocation and evaluation workflow for programmatic testing.

## Imports

In [1]:
import boto3
from IPython.display import Markdown, display
from utils import (
    EvaluationClient,
    generate_session_id,
    invoke_and_evaluate,
)

# Set your AWS Credentials

# os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
# os.environ['AWS_ACCESS_KEY_ID'] = ''
# os.environ['AWS_SECRET_ACCESS_KEY'] = ''
# os.environ['AWS_SESSION_TOKEN'] = ''


## Configuration

In [2]:
AGENT_ID = "researchappAgent-6yOZvH8hbX"
AGENT_ARN = f"arn:aws:bedrock-agentcore:us-east-1:774305571746:runtime/researchappAgent-6yOZvH8hbX"
REGION = "us-east-1"

## Experiment Setup

In [3]:
EXPERIMENT_NAME = "researcher_agent_evaluation"

# Set to None to run ALL 13 evaluators (comprehensive mode)
# Or set to a specific list like FLEXIBLE_EVALUATORS to run only those
EXPERIMENT_EVALUATORS = None  # Runs all 13 evaluators

EXPERIMENT_SCOPE = "session"  # Ignored when EXPERIMENT_EVALUATORS = None
EXPERIMENT_DELAY = 120 # atleast 120 seconds for the traces to hit AgentCore observability

planned_session = generate_session_id() 

# metadata is optional but you would appreciate it for tracking
EXPERIMENT_PROMPTS = [
    {"prompt": "Conduct a comprehensive analysis of trastuzumab (Herceptin) mechanism of action, and resistance mechanisms.\nI need:\n1. HER2 protein structure and binding sites\n2. Downstream signaling pathways affected\n3. Known resistance mechanisms from clinical data and adverse events from OpenFDA data\n4. Current clinical trials investigating combination therapies\n5. Biomarkers for treatment response prediction\n\nPlease query relevant databases to provide a comprehensive research report.", "session_id": "", "metadata": {"category": "oncology"}},
    {"prompt": "Analyze the clinical significance of BRCA1 variants in breast cancer risk and treatment response.\nPlease investigate:\n1. Population frequencies of pathogenic BRCA1 variants\n2. Clinical significance and pathogenicity classifications\n3. Associated cancer risks and penetrance estimates\n4. Treatment implications (PARP inhibitors, platinum agents)\n5. Current clinical trials for BRCA1-positive patients\n\nUse multiple databases to provide comprehensive evidence.", "session_id": "", "metadata": {"category": "genetics"}},
    {"prompt": "Investigate the PI3K/AKT/mTOR pathway in cancer and identify potential therapeutic targets.\nResearch focus:\n1. Key proteins and their interactions in the pathway\n2. Pathway alterations in different cancer types\n3. Current therapeutic agents targeting this pathway\n4. Resistance mechanisms and combination strategies\n5. Biomarkers for pathway activation and drug response\n\nSynthesize data from pathway, protein, and clinical databases.", "session_id": "", "metadata": {"category": "pathways"}},
    #{"prompt": "Hello, can you help me with math?", "session_id": planned_session, "metadata": {"turn": 1}},
    #{"prompt": "What is 15 * 23?", "session_id": planned_session, "metadata": {"turn": 2}},
]

## Initialize Clients

In [4]:
# Initialize AgentCore client
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)

# Initialize Evaluation client
eval_client = EvaluationClient(
    region=REGION,
)

## Run Experiment

In [5]:
FLEXIBLE_EVALUATORS = [
    "Builtin.Correctness",
    "Builtin.Faithfulness",
    "Builtin.Helpfulness",
    "Builtin.ResponseRelevance",
    "Builtin.Conciseness",
    "Builtin.Coherence",
    "Builtin.InstructionFollowing",
    "Builtin.Refusal",
    "Builtin.Harmfulness",
    "Builtin.Stereotyping"
]

SESSION_ONLY_EVALUATORS = ["Builtin.GoalSuccessRate"]

SPAN_ONLY_EVALUATORS = [
    "Builtin.ToolSelectionAccuracy",
    "Builtin.ToolParameterAccuracy"
]

In [6]:
batch_results = []

eval_count = len(EXPERIMENT_EVALUATORS) if EXPERIMENT_EVALUATORS else 13
print(f"Experiment: {EXPERIMENT_NAME} | Prompts: {len(EXPERIMENT_PROMPTS)} | Evaluators: {eval_count}\n")

for i, config in enumerate(EXPERIMENT_PROMPTS, 1):
    prompt_text = config["prompt"]
    session_id = config.get("session_id", "")
    metadata = config.get("metadata", {})
        
    try:
        returned_session_id, content, results = invoke_and_evaluate(
            agentcore_client=agentcore_client,
            eval_client=eval_client,
            agent_arn=AGENT_ARN,
            agent_id=AGENT_ID,
            region=REGION,
            prompt=prompt_text,
            experiment_name=EXPERIMENT_NAME,
            session_id=session_id,
            metadata=metadata,
            evaluators=EXPERIMENT_EVALUATORS,
            scope=EXPERIMENT_SCOPE,
            delay=EXPERIMENT_DELAY,
            flexible_evaluators=FLEXIBLE_EVALUATORS,
            session_only_evaluators=SESSION_ONLY_EVALUATORS,
            span_only_evaluators=SPAN_ONLY_EVALUATORS
        )
        
        if content:
            display(Markdown(str(content[0])))
        
        batch_results.append({
            "session_id": returned_session_id,
            "prompt": prompt_text,
            "results": results
        })
                
    except Exception as e:
        print(f"Error: {e}\n")
        batch_results.append({"prompt": prompt_text, "error": str(e)})

Experiment: researcher_agent_evaluation | Prompts: 3 | Evaluators: 13



2026-02-13 13:22:31,366 - cloudwatch_client - INFO - Fetching session data for: bae4629c-a1a8-4963-a126-7f8397569dff
2026-02-13 13:22:31,367 - cloudwatch_client - INFO - Querying spans for session: bae4629c-a1a8-4963-a126-7f8397569dff (agent: researchappAgent-6yOZvH8hbX)
2026-02-13 13:22:34,867 - cloudwatch_client - INFO - Found 189 spans for session bae4629c-a1a8-4963-a126-7f8397569dff
2026-02-13 13:22:34,868 - cloudwatch_client - INFO - Querying runtime logs for 1 traces
2026-02-13 13:22:38,566 - cloudwatch_client - INFO - Found 302 runtime logs across 1 traces
2026-02-13 13:22:38,567 - cloudwatch_client - INFO - Session data retrieved: 189 spans, 1 traces, 302 runtime logs


Found 189 spans across 1 traces in session
✅ Timestamp normalization applied: 209 log events normalized
Sending 375 items (375 spans [355 with gen_ai attrs], 209 log events) to evaluation API
Input saved to: evaluation_input/input_bae4629c-a1a8-49_20260213_132238.json
Output saved to: evaluation_output/output_bae4629c-a1a8-49_20260213_132258.json
Found 1 evaluation output file(s)
Found 1 input file(s) with trace data
Dashboard data generated: 1 session(s), 10 evaluation(s)


2026-02-13 13:22:58,907 - cloudwatch_client - INFO - Fetching session data for: bae4629c-a1a8-4963-a126-7f8397569dff
2026-02-13 13:22:58,907 - cloudwatch_client - INFO - Querying spans for session: bae4629c-a1a8-4963-a126-7f8397569dff (agent: researchappAgent-6yOZvH8hbX)


Opening dashboard in browser: evaluation_dashboard.html


...

Opening dashboard in browser: evaluation_dashboard.html


2026-02-13 13:27:57,627 - cloudwatch_client - INFO - Found 152 spans for session 560285c9-a2cc-4cbd-b814-72d1d52fdad3
2026-02-13 13:27:57,628 - cloudwatch_client - INFO - Querying runtime logs for 1 traces
2026-02-13 13:28:01,275 - cloudwatch_client - INFO - Found 336 runtime logs across 1 traces
2026-02-13 13:28:01,276 - cloudwatch_client - INFO - Session data retrieved: 152 spans, 1 traces, 336 runtime logs


Found 152 spans across 1 traces in session
Found 10 tool execution spans for evaluation
Evaluation target: spanIds = ['723f9a0bdf78a5fa', 'feafb1d494405b4f', 'c654d9adbc06c9f9', 'adc04aeb3394ae1f', '5c77064e2f562467', '6638ec356d9ca4f7', '91d341ae3941a4b3', '2934e78b750b8b5c', '0638b60a6bf60c80', 'c43d05994b51c44b']
✅ Timestamp normalization applied: 233 log events normalized
Sending 364 items (364 spans [336 with gen_ai attrs], 233 log events) to evaluation API
Input saved to: evaluation_input/input_560285c9-a2cc-4c_20260213_132801.json
Output saved to: evaluation_output/output_560285c9-a2cc-4c_20260213_132818.json
Found 6 evaluation output file(s)
Found 6 input file(s) with trace data
Dashboard data generated: 2 session(s), 62 evaluation(s)
Opening dashboard in browser: evaluation_dashboard.html


"I'll conduct"

Error: Read timeout on endpoint URL: "None"

